In [2]:
"""
Text Duplication Features Extraction (OPTIMIZED - Fast Version)
================================================================
极速优化版 - 避免慢速pairwise比较

优化策略:
1. 主要用exact match（哈希加速，O(n)）
2. Fuzzy match只采样少量pairs（避免O(n²)）
3. 使用简化相似度算法
4. 预期速度提升: 10-100倍

输入: reddit_wsb_for_network.csv
输出: text_duplication_features_5min.parquet

核心特征（6列）:
1. exact_duplicate_rate - 完全相同文本比例 (快速)
2. fuzzy_duplicate_rate - 采样估计的相似文本比例 (快速)
3. unique_text_ratio - 独特文本比例 (快速)
4. avg_text_similarity - 采样估计的平均相似度 (快速)
5. max_duplicate_count - 单个文本最大重复次数 (快速)
6. duplicate_cluster_size - 重复文本簇大小 (快速)
"""

import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
import hashlib

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
DATA_FILE = 'reddit_wsb_for_network.csv'

# 优化参数
MAX_SAMPLE_PAIRS = 100  # 最多采样100个pairs做fuzzy比较（大幅减少计算）
SIMILARITY_THRESHOLD = 0.8

# ============================================================
# 优化的核心函数
# ============================================================

def fast_text_hash(text):
    """
    快速文本哈希（用于exact match）
    """
    if pd.isna(text):
        return ''
    
    # 标准化
    text = str(text).lower().strip()
    
    # 简单清理
    import re
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def fast_similarity(text1, text2):
    """
    快速相似度计算（Jaccard on words）
    比SequenceMatcher快很多
    """
    if not text1 or not text2:
        return 0
    
    # 分词
    words1 = set(text1.split())
    words2 = set(text2.split())
    
    if len(words1) == 0 and len(words2) == 0:
        return 0
    
    # Jaccard similarity
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    
    if union == 0:
        return 0
    
    return intersection / union


def extract_text_duplication_features_fast(window_data):
    """
    快速版本 - 避免O(n²)复杂度
    """
    
    if len(window_data) == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    texts = window_data['text'].tolist()
    total_texts = len(texts)
    
    if total_texts == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    # ========== 1. Exact Duplicates (快速哈希) ==========
    normalized_texts = [fast_text_hash(t) for t in texts]
    normalized_texts = [t for t in normalized_texts if t != '']
    
    if len(normalized_texts) == 0:
        return {
            'exact_duplicate_rate': 0,
            'fuzzy_duplicate_rate': 0,
            'unique_text_ratio': 100,
            'avg_text_similarity': 0,
            'max_duplicate_count': 0,
            'duplicate_cluster_size': 0
        }
    
    # 计数（O(n)）
    text_counts = Counter(normalized_texts)
    unique_texts = len(text_counts)
    
    # 指标
    exact_dup_rate = (1 - unique_texts / len(normalized_texts)) * 100
    unique_ratio = (unique_texts / len(normalized_texts)) * 100
    max_dup_count = max(text_counts.values()) if text_counts else 0
    duplicate_cluster_size = max(text_counts.values()) if text_counts else 0
    
    # ========== 2. Fuzzy Duplicates (采样) ==========
    # 只采样少量pairs，避免O(n²)
    
    if len(normalized_texts) > 1:
        # 随机采样pairs
        n = len(normalized_texts)
        
        # 最多采样MAX_SAMPLE_PAIRS个pairs
        if n * (n - 1) // 2 > MAX_SAMPLE_PAIRS:
            # 大数据集：采样
            num_samples = MAX_SAMPLE_PAIRS
            indices = np.random.choice(n, size=min(num_samples * 2, n), replace=False)
            
            similarities = []
            fuzzy_matches = 0
            
            for i in range(0, len(indices) - 1, 2):
                if i + 1 < len(indices):
                    idx1, idx2 = indices[i], indices[i + 1]
                    sim = fast_similarity(normalized_texts[idx1], normalized_texts[idx2])
                    similarities.append(sim)
                    
                    if sim >= SIMILARITY_THRESHOLD:
                        fuzzy_matches += 1
            
            avg_similarity = np.mean(similarities) if similarities else 0
            fuzzy_dup_rate = (fuzzy_matches / len(similarities) * 100) if similarities else 0
            
        else:
            # 小数据集：完整计算
            similarities = []
            fuzzy_matches = 0
            
            for i in range(n):
                for j in range(i + 1, n):
                    sim = fast_similarity(normalized_texts[i], normalized_texts[j])
                    similarities.append(sim)
                    
                    if sim >= SIMILARITY_THRESHOLD:
                        fuzzy_matches += 1
            
            avg_similarity = np.mean(similarities) if similarities else 0
            fuzzy_dup_rate = (fuzzy_matches / len(similarities) * 100) if similarities else 0
    else:
        avg_similarity = 0
        fuzzy_dup_rate = 0
    
    return {
        'exact_duplicate_rate': exact_dup_rate,
        'fuzzy_duplicate_rate': fuzzy_dup_rate,
        'unique_text_ratio': unique_ratio,
        'avg_text_similarity': avg_similarity,
        'max_duplicate_count': max_dup_count,
        'duplicate_cluster_size': duplicate_cluster_size
    }


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("TEXT DUPLICATION FEATURES EXTRACTION (OPTIMIZED - FAST VERSION)")
    print("="*70)
    
    print("\nOptimizations:")
    print(f"  ✓ Exact match: Hash-based (O(n) instead of O(n²))")
    print(f"  ✓ Fuzzy match: Sampled (max {MAX_SAMPLE_PAIRS} pairs)")
    print(f"  ✓ Similarity: Jaccard on words (faster than SequenceMatcher)")
    print(f"  Expected speedup: 10-100x")
    
    # ========== Step 1: 加载数据 ==========
    print("\nStep 1: Loading Reddit data...")
    
    try:
        reddit_df = pd.read_csv(DATA_FILE)
        reddit_df['timestamp'] = pd.to_datetime(reddit_df['timestamp'])
        
        print(f"  ✓ Loaded {len(reddit_df):,} items")
        
        if 'text' not in reddit_df.columns:
            print(f"  ✗ No 'text' column found!")
            return
        
    except FileNotFoundError:
        print(f"  ✗ {DATA_FILE} not found!")
        return
    
    # ========== Step 2: 创建时间窗口 ==========
    print("\nStep 2: Creating time windows...")
    
    reddit_df['time_window'] = reddit_df['timestamp'].dt.floor(f'{TIME_WINDOW}min')
    time_windows = sorted(reddit_df['time_window'].unique())
    
    print(f"  ✓ Created {len(time_windows):,} time windows")
    
    # ========== Step 3: 逐窗口提取特征 ==========
    print("\nStep 3: Extracting text duplication features (FAST)...")
    
    features_list = []
    
    for i, window_time in enumerate(time_windows):
        if i % 100 == 0:
            elapsed = (datetime.now() - start_time).total_seconds()
            rate = i / elapsed if elapsed > 0 else 0
            eta = (len(time_windows) - i) / rate if rate > 0 else 0
            print(f"  Processing {i+1:,}/{len(time_windows):,} ({rate:.0f} windows/sec, ETA: {eta:.0f}s)...", end='\r')
        
        # 获取该窗口的数据
        window_data = reddit_df[reddit_df['time_window'] == window_time].copy()
        
        # 提取特征（快速版）
        window_features = extract_text_duplication_features_fast(window_data)
        window_features['timestamp'] = window_time
        
        features_list.append(window_features)
    
    print(f"\n  ✓ Extracted features for {len(features_list):,} windows")
    
    # 转换为DataFrame
    duplication_df = pd.DataFrame(features_list)
    
    # ========== Step 4: 统计分析 ==========
    print("\nStep 4: Feature statistics...")
    
    print(f"\n  Exact duplication:")
    print(f"    Mean rate: {duplication_df['exact_duplicate_rate'].mean():.1f}%")
    print(f"    Max rate: {duplication_df['exact_duplicate_rate'].max():.1f}%")
    high_exact = (duplication_df['exact_duplicate_rate'] > 10).sum()
    print(f"    Windows with >10%: {high_exact:,}")
    
    print(f"\n  Fuzzy duplication (sampled):")
    print(f"    Mean rate: {duplication_df['fuzzy_duplicate_rate'].mean():.1f}%")
    print(f"    Max rate: {duplication_df['fuzzy_duplicate_rate'].max():.1f}%")
    high_fuzzy = (duplication_df['fuzzy_duplicate_rate'] > 15).sum()
    print(f"    Windows with >15%: {high_fuzzy:,}")
    
    print(f"\n  Duplicate clusters:")
    print(f"    Mean max count: {duplication_df['max_duplicate_count'].mean():.1f}")
    print(f"    Max count: {duplication_df['max_duplicate_count'].max()}")
    large_clusters = (duplication_df['duplicate_cluster_size'] > 5).sum()
    print(f"    Windows with clusters >5: {large_clusters:,}")
    
    # ========== Step 5: 保存 ==========
    print("\nStep 5: Saving...")
    
    duplication_df.to_parquet('text_duplication_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: text_duplication_features_5min.parquet")
    print(f"  Shape: {duplication_df.shape}")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    
    print(f"\nTime windows: {len(duplication_df):,}")
    print(f"Feature columns: {len(duplication_df.columns)}")
    print(f"Runtime: {duration:.1f} seconds ({len(time_windows)/duration:.1f} windows/sec)")
    
    print(f"\nCoordination signals:")
    total_suspicious = len(duplication_df[
        (duplication_df['exact_duplicate_rate'] > 10) |
        (duplication_df['fuzzy_duplicate_rate'] > 15) |
        (duplication_df['duplicate_cluster_size'] > 5)
    ])
    
    print(f"  Suspicious windows: {total_suspicious:,} ({total_suspicious/len(duplication_df)*100:.1f}%)")
    
    print("\n✓ Text duplication feature extraction complete (FAST)!")
    
    if total_suspicious > 0:
        print(f"\n✓ Found coordination signals despite low diagnostic score!")
    else:
        print(f"\n⚠️  Limited signals (consistent with diagnostic score 5.7)")


if __name__ == "__main__":
    main()


TEXT DUPLICATION FEATURES EXTRACTION (OPTIMIZED - FAST VERSION)

Optimizations:
  ✓ Exact match: Hash-based (O(n) instead of O(n²))
  ✓ Fuzzy match: Sampled (max 100 pairs)
  ✓ Similarity: Jaccard on words (faster than SequenceMatcher)
  Expected speedup: 10-100x

Step 1: Loading Reddit data...
  ✓ Loaded 1,606,093 items

Step 2: Creating time windows...
  ✓ Created 77,267 time windows

Step 3: Extracting text duplication features (FAST)...
  Processing 77,201/77,267 (351 windows/sec, ETA: 0s).....
  ✓ Extracted features for 77,267 windows

Step 4: Feature statistics...

  Exact duplication:
    Mean rate: 0.6%
    Max rate: 68.0%
    Windows with >10%: 1,003

  Fuzzy duplication (sampled):
    Mean rate: 0.2%
    Max rate: 100.0%
    Windows with >15%: 277

  Duplicate clusters:
    Mean max count: 1.3
    Max count: 243
    Windows with clusters >5: 1,180

Step 5: Saving...
  ✓ Saved: text_duplication_features_5min.parquet
  Shape: (77267, 7)

COMPLETE

Time windows: 77,267
Feature 